# Gemma 4 Developer Agent — Kaggle Evaluation Notebook

End-to-end evaluation of the competition agent on real SWE tasks, running the actual base model (`gemma-4-31b-it-qat-w4a16-ct`) via vLLM on Kaggle GPUs. It mirrors the two-phase `swegemma` lifecycle documented in `HARNESS_README.md`:

- **Phase 1 (agent)**: fresh git workspace per task, the 9 sandboxed tools (`run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, `submit_patch`, `search_similar_code`, `get_code_neighbors`, `get_code_subgraph`), budget gating with continuation nudges, patch extraction via `git add -N . && git diff`.
- **Phase 2 (verification)**: fresh workspace at `base_commit` → 4-pass resilient patch application → protected test/config reset → `test_patch` application → hermetic `pytest` with JUnit XML scoring (`resolved` = exit code 0 AND every FAIL_TO_PASS / PASS_TO_PASS test passes).

## Setup

1. **GPU**: Settings → Accelerator → GPU. 2xT4 works (tensor parallel = 2); 4xL4 matches the competition hardware.
2. **Model access**: if `google/gemma-4-31b-it-qat-w4a16-ct` is gated on Hugging Face, add `HF_TOKEN` under Add-ons → Secrets. Or set `LOCAL_MODEL_DIR` / `KAGGLE_MODEL` in the config cell to an attached model copy.
3. **Submission files**: attach this repository as a Kaggle Dataset (it contains `agent.yaml`). If none is attached, an embedded snapshot of the current submission is written and used — regenerate it with `python kaggle/build_notebook.py` after editing prompts/configs.
4. **Tasks**: default is a slice of SWE-bench Lite (downloaded via `datasets`, needs internet). Point `TASKS_JSONL` at the competition's task file once published.
5. Run All. Loading the model into vLLM takes 10–25 min; each task then runs agent + verification (minutes).

## Notes

- The agent core here re-implements the documented harness protocol (tools, budgets, nudges, patch extraction, verification). If the competition publishes the real `swegemma` packages for participant-side use, prefer their CLI for exact-fidelity runs; use this notebook when they are unavailable.
- `python kaggle/selftest_local.py` (repo, CPU-only, no model) validates the agent core end-to-end with a scripted model.


In [ ]:
# ============================== CONFIGURATION ==============================
# Edit this cell, then Run All. Everything below is tunable without touching any other cell.

# ---- Model ----
HF_MODEL_ID = "google/gemma-4-31b-it-qat-w4a16-ct"   # HF repo id (competition alias)
LOCAL_MODEL_DIR = ""    # path to a local model dir (e.g. /kaggle/input/...) to skip the HF download
KAGGLE_MODEL = ""       # Kaggle Models slug for kagglehub, e.g. "google/gemma-4/transformers/..."
SERVED_NAME = "gemma-4-31b-it-qat-w4a16-ct"           # name the API is served under
MAX_MODEL_LEN = 32768
GPU_MEM_UTIL = 0.90

# ---- Submission (agent.yaml / prompts / configs) ----
SUBMISSION_DIR = ""     # e.g. /kaggle/input/<dataset>/gemma4 — auto-detected when empty

# ---- Tasks ----
TASK_SOURCE = "swe_bench_lite"   # "swe_bench_lite" | "swe_bench_verified" | "jsonl"
TASKS_JSONL = ""                 # path to a competition-format tasks.jsonl when TASK_SOURCE="jsonl"
TASK_IDS = []                    # restrict to these instance_ids (optional)
MAX_TASKS = 3                    # how many tasks to evaluate

# ---- Per-task agent budget ----
BUDGET = dict(timeout_seconds=300, max_tool_calls=100, max_time_minutes=25, max_turns=200)
USE_SUBMISSION_BUDGET = False    # True: the submission's eval_config.yaml overrides BUDGET

# ---- Runtime ----
PORT = 8000
VERIFY = True                    # False: collect patches only (faster prompt iteration)
REPO_CACHE = "/kaggle/working/repocache"    # local git object cache (cloned once per repo)
WORK_ROOT = "/kaggle/working/workspaces"    # per-task agent workspaces
VERIFY_ROOT = "/kaggle/working/verify"      # per-task verification workspaces

import os

def _gpu_info():
    try:
        import torch
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n))
        return n, names, total
    except Exception:
        return 0, [], 0

NGPU, GPU_NAMES, GPU_TOTAL = _gpu_info()
TP_SIZE = max(1, NGPU) if NGPU else 1
print(f"GPUs: {NGPU} x {GPU_NAMES or ['none']} | tensor_parallel_size={TP_SIZE}")
if NGPU and GPU_TOTAL < 20 * 1024**3:
    print("WARNING: <20GB total VRAM — a 31B QAT w4a16 model (~17GB) may not fit.")
if not (os.environ.get("HF_TOKEN") or LOCAL_MODEL_DIR or KAGGLE_MODEL):
    print("NOTE: HF_TOKEN not set. If the model repo is gated, add it under Add-ons -> Secrets -> HF_TOKEN.")


In [ ]:
# ============================== DEPENDENCIES ==============================
# vLLM pulls its own torch build (~3 GB, 5-10 min). Safe to re-run.
%pip install -q vllm openai datasets pandas
print("dependencies installed")


In [ ]:
# ============================== SUBMISSION FILES ==============================
# The runner tests exactly what the competition harness compiles: agent.yaml with
# !include'd prompts/configs. Preference order:
#   1. SUBMISSION_DIR set in the config cell
#   2. any attached Kaggle Dataset containing agent.yaml + prompts/system.md
#   3. an embedded snapshot of this repo's submission (written to ./submission)
# Regenerate the snapshot with `python kaggle/build_notebook.py` after editing prompts.

EMBEDDED_SUBMISSION = json.loads(r"""{
 "agent.yaml": "agent_class: LlmAgent\nname: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: \"Autonomous software engineering agent designed to locate, reproduce, fix, and verify repository bugs using the Gemma 4 architecture.\"\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\nskills:\n  - skills/repo_navigation\ngenerate_content_config: !include configs/sampling.yaml\n",
 "eval_config.yaml": "# Per-task evaluation budgets requested from the harness (swegemma `EvalConfig`).\n# These apply during Stage 1 inference; container setup time is excluded.\nevaluation:\n  timeout_seconds: 300    # per run_command call\n  max_tool_calls: 100     # ~15 localization + 15 reads + 20 edits + 50 verify\n  max_time_minutes: 60    # full agent session\n  max_turns: 200          # hard loop cap; nudges end stuck sessions far earlier\n",
 "configs/sampling.yaml": "temperature: 0.0\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 16384\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n",
 "prompts/system.md": "# SWE Coder \u2014 Gemma 4 Agent Protocol\n\nYou are an autonomous software engineer fixing real issues in Python repositories. Each task gives you a repository snapshot in `/workspace` and a problem statement. Locate the defect, apply a minimal patch, verify it, and submit it with `submit_patch()`.\n\n## Hard Rules\n\n1. Work only inside `/workspace`. Create ALL scratch files (repro scripts, notes) in `/tmp/` via `run_command` shell heredocs, e.g. `cat << 'EOF' > /tmp/repro.py`. `write_file` always resolves inside `/workspace`, and the final patch is extracted with `git add -N . && git diff` \u2014 any file you leave there ends up in the patch.\n2. NEVER edit test files (`test_*.py`, `tests/`, `testing/`) or test configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `tox.ini`). Grading resets them to baseline \u2014 fix the library code instead.\n3. Offline environment: no `pip install`, no downloads. All dependencies are pre-installed.\n4. Keep command output small: `pytest -q` with `-x`, `| tail -50` on long output. Output beyond 5,000 characters is truncated.\n\n## Tool Discipline\n\n1. **Localize first**: call `search_similar_code` with the exact symbol name from the issue (a class/function/method identifier, never prose). It returns the file and line in one call.\n2. **Read slices, not files**: `read_file` a 30\u201350 line window centered on the target line (`start_line = line - 10`, `end_line = line + 40`). Never read a long file from line 1.\n3. **Graph tools sparingly**: `get_code_neighbors` only to trace callers/callees of an already-located symbol. Never call `get_code_subgraph` unless tracing 3+ interdependent modules.\n4. **Edit with anchors**: in `edit_file`, include 2\u20133 unchanged surrounding lines in `old_string` so the match is unique. Re-read the target slice first and match whitespace exactly. Keep each edit under ~30 lines; split larger changes into multiple edits.\n5. **Watch the budget**: `get_status()` is free. When \u226410 tool calls remain, finalize and submit.\n\n## Workflow (per task)\n\n1. Read the problem statement and hints. Extract target symbols, expected vs. actual behavior, and any traceback.\n2. `search_similar_code(\"<Symbol>\")` \u2192 targeted `read_file` slice.\n3. Write a minimal reproduction to `/tmp/repro.py` and run it with `run_command`. It must FAIL before your fix; if it passes, the repro is wrong \u2014 refine it.\n4. Apply the minimal fix with `edit_file`.\n5. Re-run `/tmp/repro.py` (must pass), then the relevant existing tests: `python3 -m pytest <test_file>::<test> -q`.\n6. `git status && git diff` \u2014 confirm only intended source files changed.\n7. `submit_patch()` \u2014 final tool call; it ends the session.\n\nIf verification fails, iterate steps 4\u20135. Never submit unverified changes or leave untracked files in `/workspace`.\n",
 "skills/repo_navigation/SKILL.md": "---\nname: repo_navigation\ndescription: Structured methodology and sandboxed Python utilities for navigating large Python codebases, querying AST call graphs, and pinpointing issue locations.\n---\n\n# Repository Navigation Skill\n\nThis skill provides autonomous agents with structured methodologies and executable scripts to efficiently explore unfamiliar codebases, locate fault locations, and navigate call graphs without overflowing context windows.\n\n---\n\n## 1. When to Use This Skill\n\n- When starting an issue on an unfamiliar repository.\n- When the problem description references specific classes, functions, or exception messages, but the file path is unknown.\n- When an issue requires tracing caller/callee relationships across multiple modules.\n- When searching across files with high token efficiency.\n\n---\n\n## 2. Navigating the Codebase: Recommended Workflow\n\n### Step 1: Pre-Computed Graph & Embedding Lookup\n1. If the repository provides pre-computed code graphs and embeddings, use:\n   - `search_similar_code(query=\"<symbol_name>\")`: Locate candidate nodes. Use exact identifier names (e.g. `HTTPConnection` or `parse_header`), NOT free-form prose.\n   - `get_code_neighbors(node=\"<node_name>\")`: Find incoming and outgoing edges (callers, callees, imports).\n   - `get_code_subgraph(nodes=[\"<node1>\", \"<node2>\"])`: View interconnected components.\n\n### Step 2: Sandboxed Python Script Search\nIf symbol names are ambiguous or graph tools return no match, run the bundled helper script:\n```bash\npython3 skills/repo_navigation/scripts/repo_search.py --symbol <SymbolName>\n```\nor search for literal patterns / error strings:\n```bash\npython3 skills/repo_navigation/scripts/repo_search.py --pattern \"<search_text>\"\n```\nThis utility automatically ignores `.git`, `__pycache__`, build artifacts, and virtual environments, returning concise line references within the 5,000 character output limit.\n\n### Step 3: Targeted File Reading\n- Never read an entire file if only one function is relevant.\n- Use `read_file(filepath=\"...\", start_line=..., end_line=...)` around the discovered definition or usage lines.\n\n---\n\n## 3. Bundled Utilities & Resources\n\n- `scripts/repo_search.py`: Lightweight AST and text scanner designed to run within the sandbox environment.\n- `resources/navigation_patterns.md`: Deep dive into common Python repository architectures, locating reproduction hooks, and tracing exception stacks.\n",
 "skills/repo_navigation/scripts/repo_search.py": "#!/usr/bin/env python3\n\"\"\"Lightweight repository search utility for SWE Gemma agents in sandboxed environments.\n\nUsage:\n  python3 repo_search.py --symbol MyClass\n  python3 repo_search.py --pattern \"error_message\"\n  python3 repo_search.py --func my_function\n\"\"\"\n\nimport argparse\nimport ast\nimport os\nimport re\nimport sys\n\nIGNORE_DIRS = {\n    \".git\",\n    \"__pycache__\",\n    \".pytest_cache\",\n    \".mypy_cache\",\n    \"build\",\n    \"dist\",\n    \".tox\",\n    \"venv\",\n    \".venv\",\n    \"site-packages\",\n}\n\n\ndef search_ast_symbols(root_dir, symbol_name, symbol_type=None):\n    results = []\n    symbol_lower = symbol_name.lower()\n\n    for dirpath, dirnames, filenames in os.walk(root_dir):\n        dirnames[:] = [d for d in dirnames if d not in IGNORE_DIRS and not d.endswith(\".egg-info\")]\n\n        for fname in filenames:\n            if not fname.endswith(\".py\"):\n                continue\n\n            filepath = os.path.join(dirpath, fname)\n            rel_path = os.path.relpath(filepath, root_dir).replace(\"\\\\\", \"/\")\n\n            try:\n                with open(filepath, \"r\", encoding=\"utf-8\", errors=\"ignore\") as f:\n                    content = f.read()\n                tree = ast.parse(content, filename=rel_path)\n            except Exception:\n                continue\n\n            for node in ast.walk(tree):\n                matched = False\n                matched_kind = \"\"\n                name = \"\"\n\n                if isinstance(node, ast.ClassDef):\n                    name = node.name\n                    matched_kind = \"class\"\n                elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                    name = node.name\n                    matched_kind = \"def\"\n\n                if name and (name == symbol_name or name.lower() == symbol_lower):\n                    if symbol_type is None or symbol_type == matched_kind:\n                        matched = True\n\n                if matched:\n                    results.append((rel_path, node.lineno, f\"{matched_kind} {name}\"))\n\n    return results\n\n\ndef search_text_pattern(root_dir, pattern_str, is_regex=False):\n    results = []\n    flags = re.IGNORECASE\n    try:\n        regex = re.compile(pattern_str if is_regex else re.escape(pattern_str), flags)\n    except re.error as e:\n        print(f\"Error compiling regex: {e}\", file=sys.stderr)\n        return results\n\n    for dirpath, dirnames, filenames in os.walk(root_dir):\n        dirnames[:] = [d for d in dirnames if d not in IGNORE_DIRS and not d.endswith(\".egg-info\")]\n\n        for fname in filenames:\n            if fname.endswith((\".pyc\", \".so\", \".tar\", \".gz\", \".zip\", \".png\", \".jpg\")):\n                continue\n\n            filepath = os.path.join(dirpath, fname)\n            rel_path = os.path.relpath(filepath, root_dir).replace(\"\\\\\", \"/\")\n\n            try:\n                with open(filepath, \"r\", encoding=\"utf-8\", errors=\"ignore\") as f:\n                    for lineno, line in enumerate(f, start=1):\n                        if regex.search(line):\n                            results.append((rel_path, lineno, line.strip()))\n                            if len(results) >= 50:\n                                return results\n            except Exception:\n                continue\n\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Search repository symbols and patterns.\")\n    parser.add_argument(\"--root\", default=\".\", help=\"Root directory to search (default: .)\")\n    parser.add_argument(\"--symbol\", help=\"Search for class or function by symbol name\")\n    parser.add_argument(\"--pattern\", help=\"Search for text or regex pattern across files\")\n    parser.add_argument(\"--regex\", action=\"store_true\", help=\"Treat pattern as regular expression\")\n    parser.add_argument(\"--max-results\", type=int, default=30, help=\"Maximum results to return\")\n\n    args = parser.parse_args()\n\n    if not args.symbol and not args.pattern:\n        parser.print_help()\n        sys.exit(1)\n\n    if args.symbol:\n        matches = search_ast_symbols(args.root, args.symbol)\n        print(f\"=== Symbol matches for '{args.symbol}' ({len(matches)} found) ===\")\n        for path, line, desc in matches[:args.max_results]:\n            print(f\"{path}:{line}: {desc}\")\n\n    elif args.pattern:\n        matches = search_text_pattern(args.root, args.pattern, is_regex=args.regex)\n        print(f\"=== Pattern matches for '{args.pattern}' ({len(matches)} found) ===\")\n        for path, line, text in matches[:args.max_results]:\n            truncated_text = text[:100] + (\"...\" if len(text) > 100 else \"\")\n            print(f\"{path}:{line}: {truncated_text}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "skills/repo_navigation/resources/navigation_patterns.md": "# Python Repository Navigation Patterns\n\nThis resource outlines best practices and common architectural patterns for locating bugs in large Python repositories.\n\n---\n\n## 1. Common Repository Layouts\n\n### The `src/` Layout\n- Source code is placed inside `src/<package_name>/`.\n- Example: `src/urllib3/connectionpool.py`.\n- Tests are located under `tests/` or `test/`.\n\n### The Flat Layout\n- Package folder sits directly in the repository root.\n- Example: `flask/app.py` or `django/db/models/query.py`.\n- Root modules often import top-level symbols in `__init__.py`.\n\n---\n\n## 2. Navigating from Problem Descriptions to Code\n\n1. **Extract Key Identifiers**:\n   - Look for function names, class names, or specific parameter names mentioned in the issue.\n   - Look for exception types (e.g. `TypeError`, `ValueError`, `KeyError`) or exact error strings.\n2. **Search using AST / Embedding Graph First**:\n   - Query `search_similar_code(query=\"<identifier>\")` or run `repo_search.py --symbol <identifier>`.\n   - Never grep the whole workspace with unbounded output; always filter out test directories and caches.\n3. **Trace Call Stacks**:\n   - For an error originating in function `A`, use `get_code_neighbors(node=\"A\", edge_type=\"CALLS\")` to discover what calls `A` and what `A` delegates to.\n4. **Identify the Single Point of Failure**:\n   - Avoid spreading edits across multiple files unless strictly necessary.\n   - Most SWE-bench tasks require changes to fewer than 20 lines in 1\u20132 files.\n\n---\n\n## 3. Preserving Context Window\n\n- Always inspect code with `read_file(filepath=..., start_line=..., end_line=...)`.\n- Keep slice ranges to 50\u2013100 lines around the target definition.\n- If unsure of the exact lines, inspect the symbol definition line first from `repo_search.py`.\n"
}""")

import os, glob, json, yaml


def _find_submission(preferred):
    cands = []
    if preferred:
        cands.append(preferred)
    cands += sorted(glob.glob("/kaggle/input/**/agent.yaml", recursive=True))
    cands += [os.path.join(os.getcwd(), "submission", "agent.yaml")]
    for c in cands:
        d = os.path.dirname(os.path.abspath(c))
        if os.path.isfile(c) and os.path.isfile(os.path.join(d, "prompts", "system.md")):
            return d
    return None


def _make_include_loader(root):
    class Loader(yaml.SafeLoader):
        pass

    def include_constructor(loader, node):
        rel = loader.construct_scalar(node)
        if os.path.isabs(rel) or ".." in rel.replace("\\", "/").split("/"):
            raise ValueError(f"bad !include path: {rel}")
        full = os.path.abspath(os.path.join(root, rel))
        if not full.startswith(os.path.abspath(root)):
            raise ValueError("include escapes submission root")
        if rel.endswith((".md", ".txt")):
            return open(full, encoding="utf-8").read()
        if rel.endswith((".yaml", ".yml")):
            return yaml.load(open(full, encoding="utf-8"), Loader=_make_include_loader(root))
        raise ValueError(f"unsupported include type: {rel}")

    Loader.add_constructor("!include", include_constructor)
    return Loader


def load_submission(sub_dir):
    agent = yaml.load(open(os.path.join(sub_dir, "agent.yaml"), encoding="utf-8"),
                      Loader=_make_include_loader(sub_dir))
    budget = dict(BUDGET)
    ec = os.path.join(sub_dir, "eval_config.yaml")
    if os.path.isfile(ec):
        evaluation = (yaml.safe_load(open(ec, encoding="utf-8")) or {}).get("evaluation") or {}
        for k in budget:
            if k in evaluation:
                budget[k] = evaluation[k]
    return {
        "system_prompt": agent.get("instruction") or "",
        "sampling": agent.get("generate_content_config") or {},
        "budget": budget,
        "agent_name": agent.get("name"),
    }


SUBMISSION_DIR = _find_submission(SUBMISSION_DIR)
if SUBMISSION_DIR is None:
    SUBMISSION_DIR = os.path.join(os.getcwd(), "submission")
    for rel, content in EMBEDDED_SUBMISSION.items():
        p = os.path.join(SUBMISSION_DIR, rel)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        with open(p, "w", encoding="utf-8", newline="\n") as f:
            f.write(content)
    print("No attached submission found — wrote embedded snapshot to", SUBMISSION_DIR)

SUB = load_submission(SUBMISSION_DIR)
if USE_SUBMISSION_BUDGET:
    BUDGET.update(SUB["budget"])
print(f"Submission dir : {SUBMISSION_DIR}")
print(f"Agent          : {SUB['agent_name']}")
print(f"Sampling       : {SUB['sampling']}")
print(f"Per-task budget: {BUDGET}")
print("--- system prompt (first 300 chars) ---")
print(SUB["system_prompt"][:300])


In [ ]:
# ============================== vLLM SERVER ==============================
import os, subprocess, time, glob, requests


def resolve_model_path():
    if LOCAL_MODEL_DIR and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
        return LOCAL_MODEL_DIR
    # an attached dataset that contains the weights directly
    for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        d = os.path.dirname(cfg)
        if "gemma" in d.lower() and glob.glob(os.path.join(d, "*.safetensors")):
            return d
    if KAGGLE_MODEL:
        import kagglehub
        return kagglehub.model_download(KAGGLE_MODEL)
    return HF_MODEL_ID


def start_server(model, tool_parser="gemma"):
    cmd = ["vllm", "serve", model,
           "--served-model-name", SERVED_NAME,
           "--max-model-len", str(MAX_MODEL_LEN),
           "--tensor-parallel-size", str(TP_SIZE),
           "--gpu-memory-utilization", str(GPU_MEM_UTIL),
           "--port", str(PORT),
           "--disable-log-requests"]
    if tool_parser:
        cmd += ["--enable-auto-tool-choice", "--tool-call-parser", tool_parser]
    logf = open("/kaggle/working/vllm.log", "w")
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)


def wait_ready(proc, timeout_s=2400):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return False
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(15)
    return False


def _log_tail(n=25):
    try:
        with open("/kaggle/working/vllm.log", errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except Exception:
        return "(no log)"


subprocess.run(["bash", "-c", "pkill -f 'vllm serve' 2>/dev/null; sleep 2; true"])

MODEL_PATH = resolve_model_path()
print("Serving model from:", MODEL_PATH)
SERVER_PROC = start_server(MODEL_PATH, "gemma")
if not wait_ready(SERVER_PROC):
    print("server did not come up with the 'gemma' tool-call parser. Log tail:\n", _log_tail())
    if SERVER_PROC.poll() is None:
        SERVER_PROC.kill()
    print("retrying without tool-call flags (client falls back to text-protocol parsing)...")
    SERVER_PROC = start_server(MODEL_PATH, None)
    assert wait_ready(SERVER_PROC), f"vLLM never became healthy. Log tail:\n{_log_tail(60)}"
print("vLLM is healthy.")
subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=memory.used,memory.total --format=csv 2>/dev/null || true"])


In [ ]:
# ============================== TASKS ==============================
import glob, json


def _parse_list(v):
    if not v:
        return []
    if isinstance(v, list):
        return [str(x) for x in v]
    try:
        return [str(x) for x in json.loads(v)]
    except Exception:
        return []


def load_tasks():
    if TASK_SOURCE in ("swe_bench_lite", "swe_bench_verified"):
        from datasets import load_dataset
        name = {"swe_bench_lite": "princeton-nlp/SWE-bench_Lite",
                "swe_bench_verified": "princeton-nlp/SWE-bench_Verified"}[TASK_SOURCE]
        rows = list(load_dataset(name, split="test"))
    else:
        path = TASKS_JSONL or next(iter(glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)), None)
        assert path, "TASKS_JSONL is not set and no tasks.jsonl was found under /kaggle/input"
        rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    tasks = []
    for r in rows:
        tasks.append(dict(
            instance_id=r["instance_id"], repo=r["repo"], base_commit=r["base_commit"],
            problem_statement=r["problem_statement"], hints_text=r.get("hints_text") or "",
            test_patch=r.get("test_patch") or "",
            fail_to_pass=_parse_list(r.get("fail_to_pass")),
            pass_to_pass=_parse_list(r.get("pass_to_pass"))))
    if TASK_IDS:
        wanted = set(TASK_IDS)
        tasks = [t for t in tasks if t["instance_id"] in wanted]
    return tasks[:MAX_TASKS]


TASKS = load_tasks()
print(f"{len(TASKS)} task(s) loaded:")
for t in TASKS:
    print(f"  - {t['instance_id']}  ({t['repo']} @ {t['base_commit'][:12]})")


In [ ]:
# @CORE
# ============================== AGENT CORE: code index + sandboxed tools ==============================
# Faithful re-implementation of swegemma.tools: budget gating, output caps, JSON results.
import ast, difflib, json, os, re, subprocess, time

EXCLUDE_DIRS = {".git", "__pycache__", ".pytest_cache", ".mypy_cache", "build", "dist",
                ".tox", "venv", ".venv", "node_modules"}
GIT_EXCLUDE = "__pycache__/\n*.pyc\n.pytest_cache/\n*.egg-info/\nbuild/\ndist/\n.coverage\n"


def _j(d):
    return json.dumps(d, ensure_ascii=False)


def _detect_shell():
    """Prefer bash (the notebook targets Linux/Kaggle); fall back to the platform
    shell so the CPU selftest can run on Windows/macOS machines without bash."""
    try:
        cp = subprocess.run(["bash", "-c", "echo __ok__"], capture_output=True,
                            text=True, timeout=15, errors="replace")
        if cp.returncode == 0 and "__ok__" in (cp.stdout or ""):
            return ["bash", "-c"]
    except Exception:
        pass
    return None


SHELL_CMD = _detect_shell()


def _run_shell(cmd, cwd=None, timeout=900):
    """Run a shell command; returns CompletedProcess. In the non-bash fallback,
    'VAR=value cmd' prefixes are converted to environment variables."""
    if SHELL_CMD is not None:
        return subprocess.run(SHELL_CMD + [cmd], cwd=cwd, capture_output=True,
                              text=True, errors="replace", timeout=timeout)
    env = dict(os.environ)
    m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)=(\S*)\s+(.*)$", cmd, re.S)
    if m:
        env[m.group(1)] = m.group(2)
        cmd = m.group(3)
    return subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True,
                          text=True, errors="replace", timeout=timeout, env=env)


def _ok(**kw):
    return {"status": "ok", **kw}


def _err(error_type, message, **details):
    return {"status": "error", "error_type": error_type, "error_message": message, "details": details}


class CodeIndex:
    """Lightweight AST symbol index standing in for the harness's pre-computed graph/embeddings."""

    def __init__(self, root):
        self.root = root
        self.symbols = []   # {name, kind, file, line, snippet}
        self.py_files = []
        self._build()

    def _walk(self):
        for dirpath, dirnames, filenames in os.walk(self.root):
            dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIRS]
            for fn in sorted(filenames):
                if fn.endswith(".py") and not fn.endswith(".pyc"):
                    yield os.path.join(dirpath, fn)

    def _build(self):
        for path in self._walk():
            rel = os.path.relpath(path, self.root).replace("\\", "/")
            self.py_files.append(rel)
            try:
                src = open(path, encoding="utf-8", errors="replace").read()
                tree = ast.parse(src)
            except Exception:
                continue
            lines = src.splitlines()
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    self.symbols.append({
                        "name": node.name,
                        "kind": "class" if isinstance(node, ast.ClassDef) else "function",
                        "file": rel, "line": node.lineno,
                        "snippet": "\n".join(lines[node.lineno - 1: node.lineno - 1 + 12])})

    def resolve(self, node):
        """Resolve a short symbol name to matching index entries (exact -> suffix -> case-insensitive)."""
        n = (node or "").strip()
        if not n:
            return []
        short = n.rsplit(".", 1)[-1]
        syms = [s for s in self.symbols if s["name"] == short or s["name"] == n]
        if not syms:
            ln, lshort = n.lower(), short.lower()
            syms = [s for s in self.symbols
                    if s["name"].lower() == lshort or s["name"].lower().endswith("." + lshort)]
        return syms


class SwegemmaContext:
    """The 9 sandboxed tools bound to one task workspace, with budgets and JSON outputs."""

    def __init__(self, workspace_dir, budget, index=None):
        self.workspace = os.path.abspath(workspace_dir)
        self.budget = budget
        self.tool_calls_used = 0
        self.tool_errors = 0
        self.turns_used = 0
        self.start_time = time.time()
        self.patch_submitted = False
        self.submitted_patch = ""
        self.index = index if index is not None else CodeIndex(self.workspace)

    # ---------- budget plumbing ----------
    def remaining_seconds(self):
        return self.budget["max_time_minutes"] * 60 - (time.time() - self.start_time)

    def _remaining_calls(self):
        return self.budget["max_tool_calls"] - self.tool_calls_used

    def _gate(self):
        if self._remaining_calls() <= 0:
            return _err("BudgetExceeded",
                        "tool call budget exhausted; finalize and call submit_patch")
        if self.remaining_seconds() <= 0:
            return _err("BudgetExceeded",
                        "time budget exhausted; finalize and call submit_patch")
        return None

    # ---------- path resolution ----------
    def _resolve(self, filepath):
        p = (filepath or "").strip().replace("\\", "/")
        if p.startswith("/workspace/"):
            p = p[len("/workspace/"):]
        elif p.startswith("/"):
            p = p.lstrip("/")
        if not p:
            raise ValueError("filepath is empty")
        if ".." in p.split("/"):
            raise ValueError("path traversal ('..') is forbidden")
        return os.path.join(self.workspace, *p.split("/"))

    # ---------- execution tools ----------
    def run_command(self, command):
        timeout = min(int(self.budget.get("timeout_seconds", 300)),
                      max(5, int(self.remaining_seconds())))
        try:
            cp = _run_shell(command, cwd=self.workspace, timeout=timeout)
        except subprocess.TimeoutExpired:
            return _err("TimeoutExceeded", f"command exceeded {timeout}s")
        except FileNotFoundError:
            return _err("CommandError", "no shell is available in this environment")
        out = (cp.stdout or "")[:5000]
        errout = (cp.stderr or "")[:5000]
        if cp.returncode == 0:
            return _ok(stdout=out, stderr=errout, exit_code=0)
        return _err("CommandError", f"command exited with code {cp.returncode}",
                    stdout=out, stderr=errout, exit_code=cp.returncode)

    def get_status(self):
        return _ok(
            time_elapsed_seconds=round(time.time() - self.start_time, 1),
            time_remaining_seconds=round(max(0, self.remaining_seconds()), 1),
            tool_calls_used=self.tool_calls_used,
            tool_calls_limit=self.budget["max_tool_calls"],
            tool_calls_remaining=max(0, self._remaining_calls()),
            turns_used=self.turns_used,
            turns_limit=self.budget["max_turns"],
            patch_submitted=self.patch_submitted)

    def submit_patch(self):
        self.submitted_patch = self._extract_diff()
        self.patch_submitted = True
        return _ok(patch_size=len(self.submitted_patch),
                   files_changed=self.submitted_patch.count("diff --git"))

    def extract_fallback_patch(self):
        """Harness fallback: if the session ends without submit_patch, diff the tree anyway."""
        if not self.patch_submitted:
            self.submitted_patch = self._extract_diff()

    def _extract_diff(self):
        cp = _run_shell("git add -N . && git diff --binary _swegemma_baseline",
                        cwd=self.workspace)
        if cp.returncode != 0 or not (cp.stdout or "").strip():
            cp = _run_shell("git add -N . && git diff --binary HEAD", cwd=self.workspace)
        return cp.stdout or ""

    # ---------- workspace file tools ----------
    def read_file(self, filepath, start_line=None, end_line=None):
        path = self._resolve(filepath)
        if not os.path.isfile(path):
            return _err("ValidationError", f"file not found: {filepath}")
        lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
        total = len(lines)
        s = int(start_line) if start_line else 1
        e = int(end_line) if end_line else total
        s = max(1, min(s, max(1, total)))
        e = max(s, min(e, total))
        if e - s + 1 > 150:
            e = s + 149
        chunk = lines[s - 1:e]
        text = "\n".join(chunk)
        truncated = e < total
        if len(text) > 10000:
            cut = text.rfind("\n", 0, 10000)
            text = text[:cut] if cut > 0 else text[:10000]
            truncated = True
            e = s + text.count("\n")
        return _ok(filepath=(filepath or "").strip().strip("/"), content=text,
                   start_line=s, end_line=e, total_lines=total, is_truncated=truncated)

    def edit_file(self, filepath, old_string, new_string, allow_multiple=False):
        path = self._resolve(filepath)
        if not os.path.isfile(path):
            return _err("FileEditError", f"file not found: {filepath}")
        raw = open(path, encoding="utf-8", errors="replace").read()
        if not raw.strip():
            return _err("FileEditError", "file is empty")
        if not old_string:
            return _err("FileEditError", "old_string must not be empty")
        had_crlf = "\r\n" in raw
        content = raw.replace("\r\n", "\n")
        try:
            out, strategy, occurrences = _apply_replacement(
                content, old_string, new_string, bool(allow_multiple))
        except _EditError as exc:
            return _err("FileEditError", str(exc))
        if had_crlf:
            out = out.replace("\n", "\r\n")
        with open(path, "w", encoding="utf-8", newline="") as f:
            f.write(out)
        diff = "\n".join(difflib.unified_diff(
            content.splitlines(), out.replace("\r\n", "\n").splitlines(),
            "a/" + (filepath or "").strip("/"), "b/" + (filepath or "").strip("/"), lineterm=""))
        return _ok(filepath=(filepath or "").strip("/"), occurrences=occurrences,
                   strategy=strategy, diff=diff[:5000], is_truncated=len(diff) > 5000)

    def write_file(self, filepath, content):
        path = self._resolve(filepath)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8", newline="") as f:
            f.write(content or "")
        return _ok(filepath=(filepath or "").strip("/"), size=len(content or ""))

    # ---------- code intelligence tools ----------
    def search_similar_code(self, query, k=3):
        q = (query or "").strip()
        if not q:
            return _err("ValidationError", "query must be a non-empty symbol name")
        scored = []
        for sym in self.index.symbols:
            name = sym["name"]
            lq, ln = q.lower(), name.lower()
            if name == q:
                score = 1.0
            elif ln == lq:
                score = 0.98
            elif name.startswith(q) or q in name:
                score = 0.85
            elif lq in sym["file"].lower():
                score = 0.7
            else:
                ratio = difflib.SequenceMatcher(None, lq, ln).ratio()
                score = ratio if ratio > 0.6 else 0.0
            if score > 0:
                scored.append((score, sym))
        scored.sort(key=lambda x: (-x[0], x[1]["file"], x[1]["line"]))
        try:
            k = max(1, min(10, int(k)))
        except Exception:
            k = 3
        results = [{"node_name": f'{s["file"]}:{s["line"]}::{s["name"]}', "file": s["file"],
                    "line": s["line"], "kind": s["kind"], "similarity": round(sc, 4),
                    "code": s["snippet"][:1500]} for sc, s in scored[:k]]
        return _ok(query=q, results=results, count=len(results))

    def get_code_neighbors(self, node, edge_type=None, max_neighbors=10):
        syms = self.index.resolve(node)
        if not syms:
            return _err("NodeNotFound", f"symbol '{node}' not found in the code index")
        try:
            cap = max(1, min(50, int(max_neighbors)))
        except Exception:
            cap = 10
        neighbors = [{"relation": "DEFINED_IN", "node": s["name"], "file": s["file"],
                      "line": s["line"]} for s in syms[:cap]]
        pat = re.compile(r"\b" + re.escape(syms[0]["name"]) + r"\b")
        calls = 0
        for rel in self.index.py_files:
            if calls >= cap:
                break
            path = os.path.join(self.root_or_ws(), rel)
            try:
                src_lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
            except Exception:
                continue
            for i, line in enumerate(src_lines, 1):
                stripped = line.strip()
                if pat.search(line) and not stripped.startswith(("def " + syms[0]["name"], "class " + syms[0]["name"])):
                    neighbors.append({"relation": "CALLS", "file": rel, "line": i,
                                     "code": stripped[:120]})
                    calls += 1
                    if calls >= cap:
                        break
        return _ok(node=node, edge_type=edge_type, neighbors=neighbors,
                   count=len(neighbors))

    def root_or_ws(self):
        return self.index.root if hasattr(self.index, "root") else self.workspace

    def get_code_subgraph(self, nodes):
        if not isinstance(nodes, list) or not nodes:
            return _err("ValidationError", "nodes must be a non-empty list of symbol names")
        names = [str(n) for n in nodes][:10]
        syms = []
        for n in names:
            syms.extend(self.index.resolve(n)[:3])
        edges = []
        for a in syms:
            for b in syms:
                if a is b or a["name"] == b["name"]:
                    continue
                try:
                    src = open(os.path.join(self.root_or_ws(), a["file"]),
                               encoding="utf-8", errors="replace").read().splitlines()
                except Exception:
                    continue
                region = "\n".join(src[a["line"] - 1:a["line"] + 200])
                if re.search(r"\b" + re.escape(b["name"]) + r"\b", region):
                    edges.append({"from": a["name"], "to": b["name"], "type": "CALLS",
                                  "file": a["file"]})
                    if len(edges) >= 20:
                        break
        return _ok(nodes=[s["name"] for s in syms], edges=edges,
                   node_count=len(syms), edge_count=len(edges))

    # ---------- dispatch ----------
    def execute(self, name, args):
        """Run one tool call: budget-gate, execute, serialize JSON, inject budget warning."""
        if not isinstance(args, dict):
            args = {}
        if name not in TOOL_NAMES:
            self.tool_errors += 1
            return _j(_err("ValidationError", f"unknown tool '{name}'"))
        gate = self._gate()
        if gate is not None:
            return _j(gate)
        if name in GATED_TOOLS:
            self.tool_calls_used += 1
        try:
            result = getattr(self, name)(**args)
        except TypeError as e:
            self.tool_errors += 1
            return _j(_err("ValidationError", f"bad arguments for {name}: {e}"))
        except Exception as e:
            self.tool_errors += 1
            return _j(_err("ToolError", f"{type(e).__name__}: {e}"))
        if result.get("status") == "error" and result.get("error_type") != "CommandError":
            self.tool_errors += 1
        if name in GATED_TOOLS and self.tool_calls_used >= 20 and 0 < self._remaining_calls() <= 10:
            result["budget_warning"] = (f"Only {self._remaining_calls()} tool call(s) remaining "
                                        f"({self.tool_calls_used}/{self.budget['max_tool_calls']}). "
                                        "Finalize your edits and call submit_patch soon.")
        return _j(result)


class _EditError(Exception):
    pass


def _apply_replacement(content, old, new, allow_multiple):
    """3-tier resilient matching (mirrors adk-eval-core apply_replacement).
    Returns (new_content, strategy, occurrences); raises _EditError on failure."""
    n = content.count(old)
    if n:
        if n > 1 and not allow_multiple:
            raise _EditError(f"old_string matches {n} locations; add surrounding context "
                             "lines or pass allow_multiple=true")
        return (content.replace(old, new) if allow_multiple else content.replace(old, new, 1),
                "exact", n)
    # Tier 2: line-trimmed flexible match with re-indentation
    old_lines = old.split("\n")
    c_lines = content.split("\n")
    o_stripped = [l.strip() for l in old_lines]
    hits = [i for i in range(len(c_lines) - len(o_stripped) + 1)
            if [l.strip() for l in c_lines[i:i + len(o_stripped)]] == o_stripped]
    if hits:
        if len(hits) > 1 and not allow_multiple:
            raise _EditError(f"old_string (flexible match) hits {len(hits)} locations; "
                             "add surrounding context lines")
        base = re.match(r"[ \t]*", c_lines[hits[0]]).group(0)
        oldf = re.match(r"[ \t]*", old_lines[0]).group(0)
        reb = []
        for l in new.split("\n"):
            reb.append(base + l[len(oldf):] if (l.strip() and l.startswith(oldf)) else l)
        out, i, applied = [], 0, 0
        while i <= len(c_lines) - len(o_stripped):
            if [l.strip() for l in c_lines[i:i + len(o_stripped)]] == o_stripped:
                out.extend(reb)
                i += len(o_stripped)
                applied += 1
            else:
                out.append(c_lines[i])
                i += 1
        out.extend(c_lines[i:])
        return "\n".join(out), "flexible", applied
    # Tier 3: whitespace-insensitive regex match over code delimiters
    toks = [t for t in re.split(r"([()\[\]{}:>=<])|\s+", old) if t]
    pat = re.compile(r"\s*".join(re.escape(t) for t in toks))
    matches = list(pat.finditer(content))
    if matches:
        if len(matches) > 1 and not allow_multiple:
            raise _EditError(f"old_string (regex match) hits {len(matches)} locations")
        if allow_multiple:
            return pat.sub(lambda _: new, content), "regex", len(matches)
        m = matches[0]
        return content[:m.start()] + new + content[m.end():], "regex", 1
    raise _EditError("old_string not found with exact, flexible, or regex matching")


In [ ]:
# @CORE
# ============================== AGENT CORE: chat client + tool specs + runner ==============================
# Mirrors swegemma's agent_runner: task prompt, native/text tool-call extraction,
# continuation nudges, budget-aware loop, fallback patch extraction.
import json, os, re, time


def _spec(name, desc, params):
    return {"type": "function",
            "function": {"name": name, "description": desc, "parameters": params}}


TOOL_SPECS = [
    _spec("run_command",
          "Execute a shell command in the workspace (bash). 300s timeout; output truncated to 5000 chars.",
          {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}),
    _spec("read_file",
          "Read a slice of a file (1-indexed, inclusive). Max 150 lines / 10000 chars per call.",
          {"type": "object", "properties": {"filepath": {"type": "string"},
                                           "start_line": {"type": "integer"},
                                           "end_line": {"type": "integer"}}, "required": ["filepath"]}),
    _spec("edit_file",
          "Replace old_string with new_string in an existing file. Include 2-3 unchanged "
          "surrounding lines in old_string so it matches exactly once.",
          {"type": "object", "properties": {"filepath": {"type": "string"},
                                           "old_string": {"type": "string"},
                                           "new_string": {"type": "string"},
                                           "allow_multiple": {"type": "boolean"}},
           "required": ["filepath", "old_string", "new_string"]}),
    _spec("write_file", "Create or overwrite a file inside the workspace.",
          {"type": "object", "properties": {"filepath": {"type": "string"},
                                            "content": {"type": "string"}},
           "required": ["filepath", "content"]}),
    _spec("get_status", "Return remaining time, tool-call budget, and patch submission status.",
          {"type": "object", "properties": {}, "required": []}),
    _spec("submit_patch", "Submit the current workspace diff as the final patch. Ends the session.",
          {"type": "object", "properties": {}, "required": []}),
    _spec("search_similar_code",
          "Find functions/classes by symbol name. query must be an identifier "
          "(e.g. 'HTTPConnection'), not prose.",
          {"type": "object", "properties": {"query": {"type": "string"},
                                           "k": {"type": "integer", "default": 3}},
           "required": ["query"]}),
    _spec("get_code_neighbors", "List definitions and call sites related to a symbol.",
          {"type": "object", "properties": {"node": {"type": "string"},
                                            "edge_type": {"type": "string"},
                                            "max_neighbors": {"type": "integer", "default": 10}},
           "required": ["node"]}),
    _spec("get_code_subgraph", "Return references among a list of symbols.",
          {"type": "object", "properties": {"nodes": {"type": "array", "items": {"type": "string"}}},
           "required": ["nodes"]}),
]

TOOL_NAMES = {s["function"]["name"] for s in TOOL_SPECS}
GATED_TOOLS = {"run_command", "read_file", "edit_file", "write_file",
               "search_similar_code", "get_code_neighbors", "get_code_subgraph"}


class ChatClient:
    """OpenAI-compatible client for the local vLLM server, with param degradation retries."""

    def __init__(self, base_url, model, sampling=None, api_key="EMPTY", request_timeout=900):
        self.model = model
        self.sampling = sampling or {}
        from openai import OpenAI   # lazy import keeps this cell runnable on CPU machines
        self.client = OpenAI(base_url=base_url, api_key=api_key,
                             timeout=request_timeout, max_retries=0)

    def chat(self, messages, tools=None):
        s = self.sampling
        base = {"model": self.model, "messages": messages,
                "temperature": s.get("temperature", 0.0),
                "top_p": s.get("top_p", 1.0),
                "max_tokens": int(s.get("max_output_tokens", 16384))}
        extras = {"top_k": s.get("top_k")} if s.get("top_k") is not None else None
        degradations = [  # applied progressively on 400-style rejections
            lambda kw: {**kw, "extra_body": extras},
            lambda kw: kw,                                    # drop top_k
            lambda kw: {k: v for k, v in kw.items() if k not in ("tools", "tool_choice")},
            lambda kw: {**{k: v for k, v in kw.items() if k != "max_tokens"},
                        "max_completion_tokens": base["max_tokens"]},
        ]
        last = None
        for di, decorate in enumerate(degradations):
            kw = dict(base)
            if tools:
                kw["tools"] = tools
                kw["tool_choice"] = "auto"
            kw = decorate(kw)
            for attempt in range(6):
                try:
                    return self.client.chat.completions.create(**kw)
                except Exception as e:
                    last = e
                    msg = str(e)
                    if ("400" in msg or "Bad Request" in msg or "unexpected keyword" in msg) \
                            and di < len(degradations) - 1:
                        break   # move to the next degradation
                    if attempt == 5:
                        raise
                    time.sleep(min(60.0, 2.0 * (2 ** attempt)))
        raise last


def extract_tool_calls(message):
    """Return (calls, mode). calls: list of {id, name, arguments(dict)}.
    Native server-parsed tool_calls take precedence; otherwise parse tool-call JSON
    (e.g. <|tool_call|> tagged or fenced) from the raw content text."""
    calls = []
    for tc in (getattr(message, "tool_calls", None) or []):
        fn = getattr(tc, "function", None)
        raw = getattr(fn, "arguments", "{}") if fn else "{}"
        try:
            args = json.loads(raw or "{}")
        except Exception:
            args = {"_invalid_arguments": (raw or "")[:500]}
        if not isinstance(args, dict):
            args = {"_invalid_arguments": str(args)[:500]}
        calls.append({"id": getattr(tc, "id", None) or f"call_{len(calls)}",
                      "name": getattr(fn, "name", ""), "arguments": args})
    if calls:
        return calls, "native"
    content = getattr(message, "content", None) or ""
    if not isinstance(content, str):
        content = "".join(p.get("text", "") for p in content if isinstance(p, dict))
    decoder = json.JSONDecoder()
    for i, ch in enumerate(content):
        if ch not in "{[":
            continue
        try:
            obj, _ = decoder.raw_decode(content, i)
        except ValueError:
            continue
        for item in (obj if isinstance(obj, list) else [obj]):
            if not isinstance(item, dict):
                continue
            fn = item.get("function") or {}
            name = item.get("name") or fn.get("name")
            args = item.get("arguments", fn.get("arguments", {}))
            if not name:
                continue
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    args = {"_invalid_arguments": args[:500]}
            if not isinstance(args, dict):
                args = {"_invalid_arguments": str(args)[:500]}
            calls.append({"id": f"call_{len(calls)}", "name": name, "arguments": args})
    return calls, ("text" if calls else "none")


# Continuation nudges — mirror agent_runner.py's nudge table verbatim.
NUDGE_UNCLOSED = ("Your previous response reached the token limit before the tool call finished "
                  "closing (<|tool_call|> was cut off). Do NOT repeat your prior reasoning in "
                  "thought—emit your next tool call immediately, and if calling edit_file or "
                  "write_file, split the change into smaller incremental edits.")
NUDGE_MAXTOKENS = ("Your previous response reached the token limit while thinking before a tool "
                   "call was completed. Do NOT repeat your analysis in thought—keep reasoning "
                   "under a few sentences and emit your next tool call immediately, or call "
                   "submit_patch when you have completed and verified your changes.")
NUDGE_CONTINUE = ("Please continue your work using the available tools, or call submit_patch "
                  "when you have completed and verified your changes.")


def render_system_prompt(system_prompt, task):
    out = system_prompt
    out = out.replace("{problem_description}", task["problem_statement"])
    if task.get("hints_text"):
        out = out.replace("{hints}", task["hints_text"])
    return out


def _workspace_layout(ws, max_entries=150):
    entries = []
    for dirpath, dirnames, filenames in os.walk(ws):
        dirnames[:] = [d for d in dirnames
                       if d not in (".git", "__pycache__", ".pytest_cache")
                       and not d.endswith(".egg-info")]
        for f in sorted(filenames):
            if f.endswith(".pyc"):
                continue
            rel = os.path.relpath(os.path.join(dirpath, f), ws).replace("\\", "/")
            entries.append("./" + rel)
            if len(entries) >= max_entries:
                return "\n".join(entries)
    return "\n".join(entries)


def build_task_prompt(task, budget, workspace_dir):
    """Mirror swegemma's build_agent_prompt: up to 7 structured sections."""
    secs = [f"You are evaluating a software engineering task for repository {task['repo']}.\n\n"
            f"Problem Statement:\n{task['problem_statement']}"]
    if task.get("hints_text"):
        secs.append(f"## Hints:\n{task['hints_text']}")
    secs.append("## Task Budget (Session terminates when any budget is exhausted)\n"
                f"- Time allowance: {budget['max_time_minutes']} minutes\n"
                f"- Tool calls allowance: {budget['max_tool_calls']} calls\n"
                f"- Max loop iterations: {budget['max_turns']} turns")
    secs.append("## Execution Environment Rules\n"
                f"- Single command timeout: {budget['timeout_seconds']} seconds (commands exceeding this fail without ending the session)\n"
                "- Command output limit: 5000 characters\n"
                "- File view limit: 150 lines per read_file call\n"
                "- File character limit: 10000 characters per read_file call\n"
                "- Environment is offline (no network/PyPI access). All repository and test "
                "dependencies are ALREADY pre-installed. Do NOT attempt to run pip install or "
                "download packages.")
    secs.append("## Instructions\n"
                "0. Work strictly under /workspace.\n"
                "1. Inspect existing code and conventions before editing.\n"
                "2. Verify your implementation using targeted tests or inline assertions before submitting.\n"
                "3. Call submit_patch once complete.\n"
                "4. Return a final text completion response.")
    secs.append("## Code Intelligence Tools\n"
                "This repository has pre-built code graph and embedding data. Use these tools "
                "for fast, targeted navigation:\n"
                "- search_similar_code(query): Find semantically similar functions/classes by keyword.\n"
                "- get_code_neighbors(node): Find callers, callees, and definitions related to a symbol.\n"
                "- get_code_subgraph(nodes): Get the induced subgraph for a set of symbols.")
    secs.append("## Workspace Layout\n" + _workspace_layout(workspace_dir))
    return "\n\n".join(secs)


def _compact_messages(messages, max_chars=60000, keep_recent=8):
    """Rough context management in the spirit of the harness's EventsCompactionConfig:
    elide old, oversized tool outputs once the estimated prompt gets large."""
    total = sum(len(str(m.get("content") or "")) for m in messages)
    if total <= max_chars:
        return messages
    idx = [i for i, m in enumerate(messages) if len(str(m.get("content") or "")) > 3000]
    for pos, i in enumerate(idx):
        if total <= max_chars or pos >= len(idx) - keep_recent:
            break
        total -= len(str(messages[i].get("content"))) - 45
        messages[i] = {**messages[i], "content": "[older tool output elided for context management]"}
    return messages


def run_agent_task(task, ctx, chat, system_prompt, budget, verbose=True):
    """Drive one task: model turns -> tool execution -> nudges -> patch extraction."""
    def log(*a):
        if verbose:
            print(*a, flush=True)

    messages = [{"role": "system", "content": render_system_prompt(system_prompt, task)},
                {"role": "user", "content": build_task_prompt(task, budget, ctx.workspace)}]
    trace, tokens = [], {"prompt": 0, "completion": 0}
    consecutive_nudges = 0
    while True:
        if ctx.turns_used >= budget["max_turns"]:
            log(f"[loop] turn budget exhausted ({budget['max_turns']})")
            break
        if ctx.remaining_seconds() <= 0:
            log("[loop] time budget exhausted")
            break
        _compact_messages(messages)
        resp = chat.chat(messages, tools=TOOL_SPECS)
        choice = resp.choices[0]
        msg = choice.message
        usage = getattr(resp, "usage", None)
        if usage:
            tokens["prompt"] += getattr(usage, "prompt_tokens", 0) or 0
            tokens["completion"] += getattr(usage, "completion_tokens", 0) or 0
        ctx.turns_used += 1
        calls, mode = extract_tool_calls(msg)
        if mode == "native":
            messages.append({"role": "assistant", "content": msg.content or "",
                             "tool_calls": [{"id": c["id"], "type": "function",
                                             "function": {"name": c["name"],
                                                          "arguments": json.dumps(c["arguments"])}}
                                            for c in calls]})
        else:
            messages.append({"role": "assistant", "content": msg.content or ""})
        if not calls:
            consecutive_nudges += 1
            if consecutive_nudges > 3:
                log("[loop] no usable tool calls 3 turns in a row — stopping")
                break
            content = msg.content or ""
            finish = (getattr(choice, "finish_reason", "") or "")
            if "<|tool_call" in content:
                nudge = NUDGE_UNCLOSED
            elif finish.lower() in ("length", "max_tokens"):
                nudge = NUDGE_MAXTOKENS
            else:
                nudge = NUDGE_CONTINUE
            log(f"[loop] no tool calls (finish={finish!r}) -> nudge {consecutive_nudges}/3")
            messages.append({"role": "user", "content": nudge})
            continue
        consecutive_nudges = 0
        text_mode_results = []
        for call in calls:
            log(f"[turn {ctx.turns_used}] {call['name']} {json.dumps(call['arguments'])[:160]}")
            result = ctx.execute(call["name"], call["arguments"])
            trace.append({"turn": ctx.turns_used, "tool": call["name"],
                          "arguments": call["arguments"], "result": result[:2000]})
            if mode == "native":
                messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            else:
                try:
                    text_mode_results.append({"tool": call["name"], "result": json.loads(result)})
                except Exception:
                    text_mode_results.append({"tool": call["name"], "result": result[:4000]})
        if text_mode_results:
            messages.append({"role": "user",
                             "content": "TOOL RESULTS:\n" + json.dumps(text_mode_results)[:12000]
                                        + "\nContinue with your next action, or call submit_patch."})
        if ctx.patch_submitted:
            log("[loop] patch submitted")
            break
    if not ctx.patch_submitted:
        ctx.extract_fallback_patch()
        log(f"[loop] session ended without submit_patch; fallback diff extracted "
            f"({len(ctx.submitted_patch)} bytes)")
    return {"patch": ctx.submitted_patch, "trace": trace, "turns": ctx.turns_used,
            "tool_calls": ctx.tool_calls_used, "tool_errors": ctx.tool_errors,
            "tokens": tokens, "patch_submitted": ctx.patch_submitted,
            "wall_seconds": round(time.time() - ctx.start_time, 1)}


In [ ]:
# @CORE
# ============================== WORKSPACE SETUP + VERIFICATION ==============================
# Container A: agent workspace at base_commit. Container B: pristine clone ->
# 4-pass patch apply -> protected-file reset -> test_patch -> hermetic pytest + JUnit scoring.
import os, re, shutil, subprocess, sys
import xml.etree.ElementTree as ET


def _sh(cmd, cwd=None, timeout=900):
    return _run_shell(cmd, cwd=cwd, timeout=timeout)


def _rmtree(path):
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)


def ensure_repo_cache(repo, cache_root):
    """Clone each upstream repo once; per-task workspaces clone locally from the cache."""
    d = os.path.join(cache_root, repo.replace("/", "__"))
    if not os.path.isdir(os.path.join(d, ".git")):
        os.makedirs(cache_root, exist_ok=True)
        cp = _sh(f'git clone --quiet https://github.com/{repo}.git "{d}"', timeout=3600)
        if cp.returncode != 0 or not os.path.isdir(os.path.join(d, ".git")):
            raise RuntimeError(f"git clone failed for {repo}: {(cp.stderr or cp.stdout)[-600:]}")
    return d


def _local_clone(source_dir, dest_dir, base_commit):
    if os.path.exists(dest_dir):
        _rmtree(dest_dir)
    cp = _sh(f'git clone --quiet --no-hardlinks "{source_dir}" "{dest_dir}"', timeout=1800)
    if cp.returncode != 0:
        raise RuntimeError(f"local clone failed: {(cp.stderr or '')[-600:]}")
    cp = _sh(f'git checkout --quiet --force {base_commit}', cwd=dest_dir, timeout=600)
    if cp.returncode != 0:
        raise RuntimeError(f"checkout {base_commit} failed: {(cp.stderr or '')[-600:]}")
    _sh("git clean -fdx -q", cwd=dest_dir)


def setup_agent_workspace(task, cache_root, ws_root, pip_install=True):
    cache = ensure_repo_cache(task["repo"], cache_root)
    ws = os.path.join(ws_root, task["instance_id"])
    os.makedirs(ws_root, exist_ok=True)
    _local_clone(cache, ws, task["base_commit"])
    _sh("git tag -f _swegemma_baseline", cwd=ws)
    exclude_dir = os.path.join(ws, ".git", "info")
    os.makedirs(exclude_dir, exist_ok=True)
    with open(os.path.join(exclude_dir, "exclude"), "w", encoding="utf-8", newline="\n") as f:
        f.write(GIT_EXCLUDE)
    install_log = ""
    if pip_install:
        cp = _sh("pip install -q -e .", cwd=ws, timeout=1800)
        install_log = ((cp.stdout or "") + (cp.stderr or "")).strip()[-300:]
    return ws, install_log


# ---------- patch parsing / protected paths ----------

def _patch_paths(patch):
    paths = []
    for m in re.finditer(r"^\+\+\+ b/(.+)$", patch or "", re.M):
        p = m.group(1).strip()
        if p and not p.startswith("/dev/null"):
            paths.append(p)
    return paths


def _is_protected(path):
    p = path.replace("\\", "/")
    base = p.rsplit("/", 1)[-1]
    if base in {"conftest.py", "pytest.ini", "pyproject.toml", "tox.ini", "setup.cfg",
                ".pytest.ini", "sitecustomize.py", "usercustomize.py", "_swegemma_stubs.py"} \
            or base.endswith(".pth"):
        return True
    if re.match(r"test_.*\.py$", base) or re.match(r".*_test\.py$", base):
        return True
    return any(seg in ("tests", "test", "testing") for seg in p.split("/")[:-1])


def reset_protected(ws, patch):
    """Force-reset any protected test/config files the agent touched (anti-tampering)."""
    paths = [p for p in _patch_paths(patch) if _is_protected(p)]
    for p in paths:
        _sh(f'git checkout HEAD -- "{p}"', cwd=ws)
        _sh(f'git clean -f -- "{p}"', cwd=ws)
    return paths


def apply_patch_resilient(ws, patch_text):
    """4-pass application, mirroring verify_task: git apply (-p1 variants) then GNU patch."""
    if not (patch_text or "").strip():
        return False, "empty patch"
    pf = os.path.join(ws, "_agent_patch.diff")
    with open(pf, "w", encoding="utf-8", newline="\n") as f:
        f.write(patch_text if patch_text.endswith("\n") else patch_text + "\n")
    passes = [
        "git apply --unsafe-paths -p1",
        "git apply --unsafe-paths -p1 -3",
        "git apply --unsafe-paths -p1 --ignore-space-change --ignore-whitespace",
        "git apply --unsafe-paths -p1 --recount",
        "git apply --unsafe-paths -p0",
        "patch -p1 --batch --forward -l",
        "patch -p0 --batch --forward -l",
    ]
    last = ""
    for p in passes:
        cp = _sh(f'{p} "{pf}"', cwd=ws, timeout=300)
        if cp.returncode == 0:
            os.remove(pf)
            return True, p
        last = (cp.stderr or cp.stdout or "")[-800:]
    if os.path.exists(pf):
        os.remove(pf)
    return False, last


# ---------- JUnit scoring ----------

def _parse_junit(path):
    res = {"passed": [], "failed": [], "errors": [], "skipped": []}
    if not os.path.isfile(path):
        return res
    for tc in ET.parse(path).getroot().iter("testcase"):
        cn, nm = tc.get("classname") or "", tc.get("name") or ""
        tags = {ch.tag for ch in tc}
        if "failure" in tags:
            res["failed"].append((cn, nm))
        elif "error" in tags:
            res["errors"].append((cn, nm))
        elif "skipped" in tags:
            res["skipped"].append((cn, nm))
        else:
            res["passed"].append((cn, nm))
    return res


def _test_matches(req_id, classname, name):
    parts = req_id.split("::")
    func = parts[-1].split("[")[0]
    if (name or "").split("[")[0] != func:
        return False
    file_part = parts[0].replace("\\", "/")
    if file_part.endswith(".py"):
        file_part = file_part[:-3]
    file_part = file_part.replace("/", ".")
    cn = classname or ""
    if file_part and file_part not in cn and not cn.endswith(file_part):
        return False
    if len(parts) > 2 and parts[1] not in cn:
        return False
    return True


def run_tests_and_score(vw, task, pip_install=True):
    test_files = sorted(set(_patch_paths(task["test_patch"])))
    if not test_files:
        return {"resolved": False, "classification": "verify_error",
                "details": "no test files found in test_patch"}
    if pip_install:
        _sh("pip install -q -e .", cwd=vw, timeout=1800)
    junit_path = os.path.join(vw, "_junit.xml")
    py = sys.executable.replace(os.sep, "/")
    targets = " ".join(f'"{t}"' for t in test_files)
    # NOTE: the production harness runs `python3 -s -m pytest`; we omit -s so that
    # user-site installs work when iterating on a local machine. PYTHONSAFEPATH=1
    # still prevents cwd injection into sys.path.
    cmd = (f'PYTHONSAFEPATH=1 "{py}" -m pytest {targets} --junitxml="{junit_path}" '
           f'-p no:cacheprovider -o timeout=0 -o python_classes="Test* *Test" -q')
    try:
        cp = _sh(cmd, cwd=vw, timeout=2400)
    except subprocess.TimeoutExpired:
        return {"resolved": False, "classification": "verify_error",
                "details": "pytest timed out"}
    junit = _parse_junit(junit_path)
    required = list(task["fail_to_pass"]) + list(task["pass_to_pass"])
    missing = [rid for rid in required
               if not any(_test_matches(rid, cn, nm) for cn, nm in junit["passed"])]
    resolved = cp.returncode == 0 and bool(junit["passed"]) and not missing
    return {"resolved": resolved,
            "missing_required": missing,
            "junit_counts": {k: len(v) for k, v in junit.items()},
            "pytest_exit": cp.returncode,
            "pytest_tail": ((cp.stdout or "") + (cp.stderr or ""))[-1500:],
            "classification": "resolved" if resolved else "not_resolved"}


def verify_task(task, agent_patch, source_repo_dir, verify_root, pip_install=True):
    """Phase 2: fresh workspace at base_commit, apply agent patch, reset protected
    files, apply test_patch, run hermetic pytest, score against F2P + P2P."""
    if not (agent_patch or "").strip():
        return {"resolved": False, "classification": "no_patch",
                "details": "agent produced an empty patch"}
    vw = os.path.join(verify_root, task["instance_id"])
    try:
        os.makedirs(verify_root, exist_ok=True)
        _local_clone(source_repo_dir, vw, task["base_commit"])
    except Exception as e:
        return {"resolved": False, "classification": "env_setup_failed",
                "details": f"verify workspace setup failed: {e}"}
    try:
        ok, info = apply_patch_resilient(vw, agent_patch)
        if not ok:
            return {"resolved": False, "classification": "patch_apply_failed",
                    "details": f"4-pass patch application failed: {info}"}
        reset_protected(vw, agent_patch)
        tp_ok, tp_info = apply_patch_resilient(vw, task["test_patch"])
        if not tp_ok:
            return {"resolved": False, "classification": "test_patch_failed",
                    "details": f"test_patch failed to apply (upstream drift?): {tp_info}"}
        result = run_tests_and_score(vw, task, pip_install=pip_install)
    finally:
        _rmtree(vw)
    return result


In [ ]:
# ============================== TOOL-CALL PROBE ==============================
# One cheap round-trip to confirm which mode the server + model produce:
# 'native' (server-parsed tool_calls) or 'text' (parsed from output text by the runner).
PROBE_SPEC = {"type": "function", "function": {"name": "echo",
              "description": "Echo the given text back.",
              "parameters": {"type": "object", "properties": {"text": {"type": "string"}},
                             "required": ["text"]}}}

try:
    probe_client = ChatClient(base_url=f"http://127.0.0.1:{PORT}/v1", model=SERVED_NAME,
                              sampling=SUB["sampling"])
    resp = probe_client.chat([{"role": "user", "content": 'Call the echo tool with text="probe".'}],
                             tools=[PROBE_SPEC])
    calls, mode = extract_tool_calls(resp.choices[0].message)
    print(f"Tool-call probe: mode={mode}, calls={[c['name'] for c in calls]}")
    if mode == "native":
        print("Native tool calling is active.")
    elif mode == "text":
        print("Text-protocol fallback active — tool calls are parsed from output text "
              "and results are returned as user messages.")
    else:
        print("WARNING: the model did not emit a tool call for the probe; inspect the response:")
        print((resp.choices[0].message.content or "")[:500])
except Exception as e:
    print("Probe failed:", e)


In [ ]:
# ============================== RUN EVALUATION ==============================
import json, os, time

EVAL_DIR = "/kaggle/working/eval_results"
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(VERIFY_ROOT, exist_ok=True)

chat = ChatClient(base_url=f"http://127.0.0.1:{PORT}/v1", model=SERVED_NAME,
                  sampling=SUB["sampling"])
results = []

t_all = time.time()
for i, task in enumerate(TASKS):
    tid = task["instance_id"]
    print(f"\n{'=' * 70}\n[{i + 1}/{len(TASKS)}] {tid}  ({task['repo']} @ {task['base_commit'][:12]})\n{'=' * 70}")
    rec = {"instance_id": tid, "repo": task["repo"]}
    try:
        ws, install_log = setup_agent_workspace(task, REPO_CACHE, WORK_ROOT, pip_install=True)
        if install_log:
            print("pip install log tail:", install_log[-300:])
    except Exception as e:
        rec.update(resolved=False, classification="env_setup_failed", details=str(e))
        results.append(rec)
        print("workspace setup failed:", e)
        continue
    ctx = SwegemmaContext(ws, budget=BUDGET, index=CodeIndex(ws))
    try:
        run = run_agent_task(task, ctx, chat, SUB["system_prompt"], BUDGET, verbose=True)
    except Exception as e:
        rec.update(resolved=False, classification="agent_crashed",
                   details=f"{type(e).__name__}: {e}")
        results.append(rec)
        print("agent crashed:", e)
        _rmtree(ws)
        continue
    with open(os.path.join(EVAL_DIR, f"{tid}.patch"), "w", encoding="utf-8", newline="\n") as f:
        f.write(run["patch"])
    with open(os.path.join(EVAL_DIR, f"{tid}.trace.json"), "w", encoding="utf-8") as f:
        json.dump(run["trace"], f, indent=1)
    rec.update(tool_calls=run["tool_calls"], turns=run["turns"],
               tool_errors=run["tool_errors"],
               prompt_tokens=run["tokens"]["prompt"],
               completion_tokens=run["tokens"]["completion"],
               wall_seconds=run["wall_seconds"], patch_bytes=len(run["patch"]))
    if VERIFY:
        v = verify_task(task, run["patch"], ensure_repo_cache(task["repo"], REPO_CACHE),
                        VERIFY_ROOT, pip_install=True)
        rec.update(resolved=bool(v.get("resolved")),
                   classification=v.get("classification", "?"),
                   missing_required=v.get("missing_required"),
                   details=v.get("details"))
        if v.get("pytest_tail"):
            rec["pytest_tail"] = v["pytest_tail"]
    else:
        rec.update(resolved=None, classification="not_verified")
    results.append(rec)
    print(f"\n[{tid}] classification={rec['classification']} resolved={rec.get('resolved')}")
    if rec.get("details"):
        print("details:", str(rec.get("details"))[:500])
    _rmtree(ws)   # free disk between tasks

print(f"\nAll tasks finished in {(time.time() - t_all) / 60:.1f} minutes.")


In [ ]:
# ============================== REPORT ==============================
import json, os
from collections import Counter
import pandas as pd

EVAL_DIR = "/kaggle/working/eval_results"

df = pd.DataFrame(results)
cols = [c for c in ["instance_id", "classification", "resolved", "tool_calls", "turns",
                    "tool_errors", "prompt_tokens", "completion_tokens", "wall_seconds",
                    "patch_bytes"] if c in df.columns]
print(df[cols].to_string(index=False))

resolved = sum(1 for r in results if r.get("resolved"))
setup_failures = sum(1 for r in results if r.get("classification") == "env_setup_failed")
attempted = len(results) - setup_failures
print(f"\nResolution rate: {resolved}/{len(results)} overall | "
      f"{resolved}/{attempted} excluding setup failures "
      f"({(resolved / attempted * 100 if attempted else 0):.1f}%)")
print("Classifications:", dict(Counter(r.get("classification", "?") for r in results)))
print(f"\nArtifacts in {EVAL_DIR}: per-task .patch and .trace.json, results.json, results.csv")

with open(os.path.join(EVAL_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=1, default=str)
df.to_csv(os.path.join(EVAL_DIR, "results.csv"), index=False)


## Using the results

- **Per-task artifacts**: `/kaggle/working/eval_results/<instance_id>.patch` (the submitted diff) and `<instance_id>.trace.json` (every tool call + truncated result). Summary in `results.json` / `results.csv`.
- **Iterate on the prompt**: edit `prompts/system.md` or `configs/sampling.yaml` in the repo, re-upload the repo dataset (or regenerate this notebook with `python kaggle/build_notebook.py` to refresh the embedded snapshot), and re-run. Compare `resolved`, `tool_errors`, and the trace quality between runs. Keep the system prompt small — every rule in it should be observable in the traces as avoided failure modes.
- **Faster iteration**: set `VERIFY = False` to collect patches only, and pick easy `TASK_IDS` (e.g. small repos like `marshmallow`, `sqlfluff`) when tuning prompts.
- **Ship it**: run `python package_submission.py . submission.zip` in the repo and upload `submission.zip` to the competition. Validate first with `python run_smoke_test.py .`.
